# Onboarding Task Runtime

Use this notebook to run the first onboarding/profile-digitization task, watch the JSON state file update, and inspect the final result.

Run the cells in order:
1. bootstrap the repo imports and clear the task state
2. edit the task input JSON
3. start the task and watch the state poll in real time


In [ ]:
# Cell 2: pick the profile number and edit the task input here.
import sys
from pathlib import Path
import json

profile_number = "1"
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "profiles").exists():
    repo_root = repo_root.parent.resolve()
if not (repo_root / "profiles").exists():
    raise RuntimeError("Could not locate profiles folder.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
profiles_root = repo_root / "profiles"
selected_profile_dir = profiles_root / profile_number
task_state_path = repo_root / "runtime" / "task_states" / "onboarding_task_state.json"
task_input = {
    "task_name": "onboarding_profile_digitization",
    "task_id": "onboarding-demo-001",
    "documents": [str(selected_profile_dir)],
}

print(json.dumps({
    "profile_number": profile_number,
    "selected_profile_dir": str(selected_profile_dir),
    "task_input": task_input,
    "state_path": str(task_state_path),
}, indent=2, ensure_ascii=False))


In [ ]:
# Cell 3: start the onboarding task and print the final digitized handoff.
import importlib
import json
import threading
import time

import tasks.onboarding_task as onboarding_task

onboarding_task = importlib.reload(onboarding_task)

result_box = {}

def _run_task() -> None:
    result_box["result"] = onboarding_task.run_onboarding_task(
        task_input,
        state_path=task_state_path,
        heartbeat_seconds=1.0,
        step_delay_seconds=1.0,
        verbose=False,
    )

worker = threading.Thread(target=_run_task, daemon=True)
worker.start()

while worker.is_alive():
    time.sleep(1)

final_state = onboarding_task.read_task_state(task_state_path)
final_result = result_box.get("result", {})
print(json.dumps({
    "status": final_state.get("status"),
    "phase": final_state.get("phase"),
    "step": final_state.get("step"),
    "message": final_state.get("message"),
    "progress": final_state.get("progress"),
    "missing_fields": final_state.get("missing_fields", []),
    "digitized_user": final_result.get("digitized_user", {}),
    "completeness": final_result.get("completeness", {}),
}, indent=2, ensure_ascii=False))
